# Dijital İkiz İnsan — Model Eğitimi

Tek Kaggle notebook'u önce Türkçe duygu polaritesi sınıflandırıcısını eğitir. Uygun etiketli CSV eklendiyse aynı çalıştırmada VAD ve Big Five regresyon görevleri de yürütülür.

Polarite (olumlu/nötr/olumsuz), sürekli VAD veya Big Five kişilik doğruluğu değildir. Hiçbir metrik gerçek eğitim/test tamamlanmadan raporlanmamalıdır.

## Kaggle ayarları

1. **Settings → Internet = On** yap.
2. Erişimin varsa **Accelerator = GPU** seç.
3. Duygu polaritesi için CSV eklemek gerekmez; notebook veriyi sabitlenmiş Hub sürümünden indirir.
4. VAD veya Big Five için kullanım hakkı doğrulanmış ve etiketlenmiş CSV dosyalarını **Add Data** ile ekle.

In [ ]:
import os, sys, json, math, random, hashlib, time, subprocess
from pathlib import Path
import numpy as np
import pandas as pd

try:
    gpu_check = subprocess.run(["nvidia-smi"], capture_output=True, text=True, check=False)
    if gpu_check.returncode == 0:
        print(gpu_check.stdout[:2500])
    else:
        print("GPU görünmüyor. Kaggle Settings > Accelerator bölümünü kontrol et. CPU ile eğitim çok yavaş olabilir.")
except FileNotFoundError:
    print("nvidia-smi bulunamadı; CPU/GPU durumu torch yüklendikten sonra kontrol edilecek.")
print("Notebook başlangıç kontrolleri hazır.")

In [ ]:
%pip install -q "transformers>=4.51,<5" "accelerate>=1.8" "safetensors>=0.5" "scikit-learn>=1.6"

## 1) Veri dosyalarını kontrol et

Kaggle sağ panelindeki **Add Data** ile CSV veri setlerini ekledikten sonra aşağıdaki hücre bulunan CSV dosyalarını ve kolonlarını listeler.

Her satırın `source` ve `license_or_consent` alanları gerçek kaynak ve kullanım dayanağını içermeli; `unknown`, `TBD`, `bilinmiyor`, `doğrulanmadı` gibi yer tutucular reddedilir. Eksik etiket veya kaynak bilgisi olan satırlar sessizce atılmaz, tüm görev durdurulur. Bu otomatik kontrol hukuki izni kanıtlamaz; araştırmacı şartları ayrıca incelemelidir.

In [ ]:
csv_files = sorted(Path('/kaggle/input').rglob('*.csv'))
print(f"Ekli CSV sayısı: {len(csv_files)}")
for path in csv_files:
    try:
        print(f"{path}: {list(pd.read_csv(path, nrows=0).columns)}")
    except Exception as exc:
        print(f"Okunamadı: {path} ({type(exc).__name__})")
VAD_DATASET_PATH = None
BIG5_DATASET_PATH = None

In [ ]:
TARGETS = {    'vad': ['valence', 'arousal', 'dominance'],    'big_five': ['openness', 'conscientiousness', 'extraversion', 'agreeableness', 'neuroticism'],}BASE_REQUIRED = {'sample_id', 'participant_id', 'text', 'source', 'license_or_consent'}TEXT_COLUMNS = ('sample_id', 'participant_id', 'text', 'source', 'license_or_consent')PROVENANCE_FIELDS = ('source', 'license_or_consent')UNVERIFIED_MARKERS = (    'unknown', 'unspecified', 'unverified', 'not verified', 'not confirmed',    'to be determined', 'tbd', 'todo', 'belirsiz', 'bilinmiyor',    'doğrulanmadı', 'doğrulanmadi', 'belirlenecek', 'lisans yok',    'lisans bilinmiyor',)INVALID_VALUES = {'', 'nan', 'none', 'null', 'n/a', 'na'}def choose_dataset(labels, configured_path=None):    required = BASE_REQUIRED | set(labels)    if configured_path:        path = Path(configured_path)        if not path.is_file():            raise FileNotFoundError(f"Yapılandırılan veri dosyası bulunamadı: {path}")        columns = set(pd.read_csv(path, nrows=0).columns)        missing = sorted(required - columns)        if missing:            raise ValueError(f"{path} için eksik kolonlar: {missing}")        return path    matches = []    for path in csv_files:        try:            columns = set(pd.read_csv(path, nrows=0).columns)        except Exception:            continue        if required.issubset(columns):            matches.append(path)    if not matches:        print(f"{labels} için uygun veri bulunamadı; bu görev atlanacak.")        return None    if len(matches) > 1:        raise ValueError(            f"{labels} için birden fazla uygun CSV bulundu: {[str(p) for p in matches]}. "            "Yanlış veri dosyasının seçilmemesi için ilgili DATASET_PATH değişkenine tam yolu yaz."        )    return matches[0]def validate_provenance_rows(rows, source_label='dataset'):    """Pure-Python fail-closed provenance check shared by the Kaggle data path."""    for row_number, row in enumerate(rows, start=2):        for field in PROVENANCE_FIELDS:            value = str(row.get(field) or '').strip().casefold()            if value in INVALID_VALUES:                raise ValueError(                    f"{source_label}: {field} boş/yer tutucu değer içeriyor (CSV satırı {row_number})."                )            marker = next((item for item in UNVERIFIED_MARKERS if item in value), None)            if marker is not None:                raise ValueError(                    f"{source_label}: {field} doğrulanmamış/yer tutucu bilgi içeriyor "                    f"(CSV satırı {row_number}); gerçek kaynak ve kullanım dayanağını doğrula."                )def validate_frame(frame, labels, path):    missing = sorted((BASE_REQUIRED | set(labels)) - set(frame.columns))    if missing:        raise ValueError(f"{path} eksik kolonlar: {missing}")    if frame.empty:        raise ValueError(f"{path}: CSV boş.")    # Fail closed: do not silently drop rows with missing labels or provenance.    clean = frame.copy()    for column in TEXT_COLUMNS:        if clean[column].isna().any():            raise ValueError(f"{path}: {column} boş/eksik değer içeriyor.")        clean[column] = clean[column].astype(str).str.strip()        invalid = clean[column].str.casefold().isin(INVALID_VALUES)        if clean[column].eq('').any() or invalid.any():            raise ValueError(f"{path}: {column} boş/yer tutucu değer içeriyor.")    validate_provenance_rows(clean.to_dict(orient='records'), source_label=str(path))    if clean['sample_id'].duplicated().any():        raise ValueError(f"{path}: sample_id değerleri benzersiz olmalı.")    for label in labels:        clean[label] = pd.to_numeric(clean[label], errors='coerce')        values = clean[label].to_numpy(dtype=float)        if not np.isfinite(values).all():            raise ValueError(f"{path}: {label} sayısal olmayan, eksik veya sonlu olmayan etiket içeriyor.")        if (values < 0).any() or (values > 1).any():            raise ValueError(f"{path}: hedef etiketleri 0–1 aralığında olmalı; veriyi doğrulanmış yöntemle normalize et.")    print(f"\nVeri dosyası: {path}")    print(f"Örnek sayısı: {len(clean)} | Katılımcı sayısı: {clean['participant_id'].nunique()}")    print("Kaynak/lisans alanları dolu ve yer tutucu olmayan değerlerden oluşuyor.")    print("Not: Bu kontrol hukuki kullanım hakkını doğrulamaz; veri şartlarını ayrıca incele.")    return clean.reset_index(drop=True)vad_path = choose_dataset(TARGETS['vad'], VAD_DATASET_PATH)big5_path = choose_dataset(TARGETS['big_five'], BIG5_DATASET_PATH)if vad_path is None and big5_path is None:    print("VAD/Big Five verisi eklenmedi; bu iki görev atlanacak.")vad_df = validate_frame(pd.read_csv(vad_path), TARGETS['vad'], vad_path) if vad_path else Nonebig5_df = validate_frame(pd.read_csv(big5_path), TARGETS['big_five'], big5_path) if big5_path else Noneprint("\nVAD veri seti:", vad_path if vad_path else "bu çalıştırmada atlanıyor")print("Big Five veri seti:", big5_path if big5_path else "bu çalıştırmada atlanıyor")

## 2) Katılımcı ve metin grubu bazlı train/validation/test bölmesi

Aynı katılımcının yanı sıra Unicode NFKC, büyük/küçük harf ve boşluk farkları normalize edildiğinde aynı metni paylaşan farklı katılımcılar da tek, bölünemez grupta tutulur. Train/validation/test arasında katılımcı veya normalize edilmiş metin sızıntısı görülürse işlem durur. Üçten az bağımsız katılımcı/metin grubu varsa bölme yapılamaz.

Gruplar birbirine bağlı tekrarlar nedeniyle hedeflenen oranlardan farklı büyüklükte olabilir. Bu beklenen bir sınırlamadır; sızıntıyı önlemek için oranları zorlamak yerine veri ve bağımsız grup sayısını artır.

In [ ]:
import unicodedata

def normalize_text(value):
    if value is None:
        raise ValueError("Metin boş değer içeriyor.")
    normalized = unicodedata.normalize('NFKC', str(value)).casefold()
    normalized = ' '.join(normalized.split())
    if not normalized or normalized in {'nan', 'none', 'null'}:
        raise ValueError("Metin boş değer içeriyor.")
    return normalized

def group_participants_by_shared_text(rows):
    """Keep participants connected by exact normalized text in one split."""
    parent = {}
    first_participant_by_text = {}

    def find(participant):
        while parent[participant] != participant:
            parent[participant] = parent[parent[participant]]
            participant = parent[participant]
        return participant

    def union(left, right):
        left_root, right_root = find(left), find(right)
        if left_root != right_root:
            lower, upper = sorted((left_root, right_root))
            parent[upper] = lower

    for row in rows:
        participant = str(row.get('participant_id') or '').strip()
        if not participant or participant.casefold() in {'nan', 'none', 'null'}:
            raise ValueError("Boş participant_id bulundu; bölme yapılmadı.")
        parent.setdefault(participant, participant)
        text = normalize_text(row.get('text'))
        previous = first_participant_by_text.get(text)
        if previous is None:
            first_participant_by_text[text] = participant
        else:
            union(participant, previous)

    groups_by_root = {}
    for participant in parent:
        groups_by_root.setdefault(find(participant), []).append(participant)
    return sorted(
        (sorted(group) for group in groups_by_root.values()),
        key=lambda group: group[0],
    )

def split_by_participant(frame, train_ratio=0.8, val_ratio=0.1, seed=42):
    if not (math.isfinite(train_ratio) and math.isfinite(val_ratio)):
        raise ValueError("Train/validation oranları sonlu olmalı.")
    if train_ratio <= 0 or val_ratio <= 0 or train_ratio + val_ratio >= 1:
        raise ValueError("Train ve validation oranları pozitif olmalı; toplamları 1'den küçük olmalı.")

    groups = group_participants_by_shared_text(frame.to_dict(orient='records'))
    if len(groups) < 3:
        raise ValueError(
            f"Train/validation/test için en az 3 bağımsız katılımcı/metin grubu gerekli; {len(groups)} bulundu. "
            "Aynı metin kopyalarını ayrı bölümlere ayırmak test sızıntısı oluşturabilir."
        )

    # A group is indivisible across train/validation/test.
    random.Random(seed).shuffle(groups)
    group_count = len(groups)
    n_train = max(1, min(group_count - 2, round(group_count * train_ratio)))
    n_val = max(1, min(group_count - n_train - 1, round(group_count * val_ratio)))

    assigned = {'train': set(), 'validation': set(), 'test': set()}
    for name, selected in (
        ('train', groups[:n_train]),
        ('validation', groups[n_train:n_train + n_val]),
        ('test', groups[n_train + n_val:]),
    ):
        for group in selected:
            assigned[name].update(group)

    out = {
        name: frame[frame['participant_id'].isin(participants)].copy().reset_index(drop=True)
        for name, participants in assigned.items()
    }
    for name, part in out.items():
        if part.empty:
            raise ValueError(f"{name} bölümü boş. Daha fazla bağımsız metin grubu ekle.")

    for left, right in (
        ('train', 'validation'),
        ('train', 'test'),
        ('validation', 'test'),
    ):
        if assigned[left] & assigned[right]:
            raise AssertionError(f"Katılımcı sızıntısı tespit edildi: {left}/{right}")
        left_texts = {normalize_text(value) for value in out[left]['text'].tolist()}
        right_texts = {normalize_text(value) for value in out[right]['text'].tolist()}
        shared = left_texts & right_texts
        if shared:
            raise AssertionError(
                f"Normalize edilmiş metin sızıntısı tespit edildi: {left}/{right}, ortak metin sayısı={len(shared)}"
            )

    print({
        name: {'rows': len(part), 'participants': int(part['participant_id'].nunique())}
        for name, part in out.items()
    })
    print(f"Bağımsız katılımcı/metin grubu: {group_count}")
    print("Katılımcı ve normalize edilmiş metin sızıntısı kontrolleri geçti.")
    return out

vad_split = split_by_participant(vad_df) if vad_df is not None else None
big5_split = split_by_participant(big5_df) if big5_df is not None else None

## 3) Türkçe BERT regresyon modeli

Temel encoder: `dbmdz/bert-base-turkish-cased`. Kayıp fonksiyonu MSE; raporlanan metrikler hedef bazında MAE, RMSE ve R²'dir. Bu metrikler yalnızca aşağıdaki hücreler gerçekten çalıştırılırsa üretilecektir.

In [ ]:
import torch
import platform
import transformers
from torch import nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, AutoTokenizer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

MODEL_NAME = 'dbmdz/bert-base-turkish-cased'
# Optional: pin a verified immutable Hugging Face commit SHA for repeatable runs.
MODEL_REVISION = None
MAX_LENGTH = 256
EPOCHS = 3
BATCH_SIZE = 8
LEARNING_RATE = 2e-5
SEED = 42

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Cihaz:', device)
if device.type != 'cuda':
    print('UYARI: GPU seçilmemiş. Eğitim CPU üzerinde çok yavaş olabilir; Kaggle Settings > Accelerator ayarını kontrol et.')

class TextRegressionDataset(Dataset):
    def __init__(self, frame, tokenizer, labels):
        self.frame = frame.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.labels = labels
    def __len__(self):
        return len(self.frame)
    def __getitem__(self, i):
        row = self.frame.iloc[i]
        enc = self.tokenizer(
            str(row['text']), truncation=True, padding='max_length',
            max_length=MAX_LENGTH, return_tensors='pt'
        )
        y = torch.tensor([float(row[x]) for x in self.labels], dtype=torch.float32)
        return {
            'input_ids': enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'labels': y,
        }

class TransformerRegressor(nn.Module):
    def __init__(self, model_name, output_dim, dropout=0.1, revision=None):
        super().__init__()
        model_kwargs = {'revision': revision} if revision else {}
        self.encoder = AutoModel.from_pretrained(model_name, **model_kwargs)
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Linear(self.encoder.config.hidden_size, output_dim)
    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        return self.head(self.dropout(outputs.last_hidden_state[:, 0]))

print("Model bileşenleri tanımlandı; ağırlıklar eğitim hücresi çalıştırılınca indirilecek.")

In [ ]:
def metrics_from_arrays(y_true, y_pred, labels):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.clip(np.asarray(y_pred, dtype=float), 0, 1)
    per_label = {}
    for i, label in enumerate(labels):
        true_col, pred_col = y_true[:, i], y_pred[:, i]
        r2 = None if len(np.unique(true_col)) < 2 else float(r2_score(true_col, pred_col))
        per_label[label] = {
            'mae': float(mean_absolute_error(true_col, pred_col)),
            'rmse': float(np.sqrt(mean_squared_error(true_col, pred_col))),
            'r2': r2,
        }
    return {
        'mae': float(mean_absolute_error(y_true, y_pred)),
        'rmse': float(np.sqrt(mean_squared_error(y_true, y_pred))),
        'per_label': per_label,
    }

def evaluate(model, loader, device, labels):
    if len(loader.dataset) == 0:
        raise ValueError("Değerlendirme bölümü boş; eğitim durduruldu.")
    model.eval()
    preds, ys = [], []
    with torch.no_grad():
        for batch in loader:
            pred = model(
                batch['input_ids'].to(device),
                batch['attention_mask'].to(device)
            ).detach().cpu().numpy()
            preds.append(pred)
            ys.append(batch['labels'].numpy())
    return metrics_from_arrays(np.concatenate(ys), np.concatenate(preds), labels)

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def build_split_manifest(frame, labels):
    canonical_csv = frame.to_csv(index=False, lineterminator='\n').encode('utf-8')
    return {
        'sha256': hashlib.sha256(canonical_csv).hexdigest(),
        'rows': int(len(frame)),
        'unique_sample_count': int(frame['sample_id'].nunique()),
        'unique_participant_count': int(frame['participant_id'].nunique()),
        'label_columns': list(labels),
        'distinct_source_value_count': int(frame['source'].nunique()),
        'distinct_license_or_consent_value_count': int(frame['license_or_consent'].nunique()),
    }

def train_task(split, labels, output_dir, task_name, source_path):
    if any(split[name].empty for name in ('train', 'validation', 'test')):
        raise ValueError(f"{task_name}: train/validation/test bölümlerinden biri boş.")
    started = time.time()
    tokenizer_kwargs = {'revision': MODEL_REVISION} if MODEL_REVISION else {}
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, **tokenizer_kwargs)
    train_loader = DataLoader(TextRegressionDataset(split['train'], tokenizer, labels), batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(TextRegressionDataset(split['validation'], tokenizer, labels), batch_size=BATCH_SIZE)
    test_loader = DataLoader(TextRegressionDataset(split['test'], tokenizer, labels), batch_size=BATCH_SIZE)
    model = TransformerRegressor(MODEL_NAME, len(labels), revision=MODEL_REVISION).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
    loss_fn = nn.MSELoss()
    scaler = torch.cuda.amp.GradScaler(enabled=(device.type == 'cuda'))
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    best = float('inf')

    for epoch in range(1, EPOCHS + 1):
        model.train()
        losses = []
        for batch in train_loader:
            optimizer.zero_grad(set_to_none=True)
            ids = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            y = batch['labels'].to(device)
            with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):
                pred = model(ids, mask)
                loss = loss_fn(pred, y)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            losses.append(float(loss.detach().cpu()))

        val = evaluate(model, val_loader, device, labels)
        print(
            f"{task_name} epoch={epoch} train_mse={np.mean(losses):.5f} "
            f"val_mae={val['mae']:.5f} val_rmse={val['rmse']:.5f}"
        )
        if val['mae'] < best:
            best = val['mae']
            torch.save(model.state_dict(), output_dir / 'best_model.pt')

    model.load_state_dict(torch.load(output_dir / 'best_model.pt', map_location=device))
    test = evaluate(model, test_loader, device, labels)
    tokenizer.save_pretrained(output_dir / 'tokenizer')

    # Compare with a simple baseline that predicts only train-set target means.
    # The test split is used once after model selection, never to pick a checkpoint.
    test_targets = split['test'][labels].to_numpy(dtype=float)
    train_mean = split['train'][labels].mean().to_numpy(dtype=float)
    baseline_predictions = np.repeat(
        train_mean.reshape(1, -1), repeats=len(split['test']), axis=0
    )
    baseline_metrics = metrics_from_arrays(test_targets, baseline_predictions, labels)

    dataset_manifest = {
        'hash_algorithm': 'sha256',
        'source_file_name': Path(source_path).name,
        'source_file_sha256': sha256_file(source_path),
        'splits': {
            name: build_split_manifest(part, labels)
            for name, part in split.items()
        },
    }
    report = {
        'task': task_name,
        'model': MODEL_NAME,
        'model_revision': MODEL_REVISION,
        'model_revision_pinned': bool(MODEL_REVISION),
        'labels': labels,
        'device': str(device),
        'seed': SEED,
        'epochs': EPOCHS,
        'batch_size': BATCH_SIZE,
        'learning_rate': LEARNING_RATE,
        'max_length': MAX_LENGTH,
        'train_rows': len(split['train']),
        'validation_rows': len(split['validation']),
        'test_rows': len(split['test']),
        'participants_by_split': {
            name: int(part['participant_id'].nunique())
            for name, part in split.items()
        },
        'dataset_manifest': dataset_manifest,
        'runtime': {
            'python': platform.python_version(),
            'numpy': np.__version__,
            'pandas': pd.__version__,
            'torch': str(torch.__version__),
            'transformers': transformers.__version__,
        },
        'training_seconds': round(time.time() - started, 2),
        'baseline': {
            'type': 'per_target_train_mean',
            'description': 'Her hedef için yalnızca eğitim kümesinin ortalamasını tahmin eder.',
            'test_metrics': baseline_metrics,
        },
        'test_metrics': test,
        'mae_improvement_over_train_mean_baseline': baseline_metrics['mae'] - test['mae'],
        'scientific_note': (
            'Gerçek test çalıştırılmadan rapora sonuç yazılmamalıdır. '
            'Kaynak/lisans açıklamalarının dolu olması hukuki kullanım hakkını kanıtlamaz.'
        ),
    }
    (output_dir / 'metrics.json').write_text(
        json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8'
    )
    print(f"{task_name} test metrikleri kaydedildi: {output_dir / 'metrics.json'}")
    print(json.dumps({
        'test_metrics': test,
        'baseline_test_metrics': baseline_metrics,
        'mae_improvement_over_train_mean_baseline': report['mae_improvement_over_train_mean_baseline'],
    }, ensure_ascii=False, indent=2))
    return report

## 4) Eğitim

Veri kontrolleri geçen görevler ayrı ayrı eğitilir. VAD verisi yoksa VAD hücresi bunu bildirip atlar; Big Five verisi yoksa Big Five hücresi bunu bildirip atlar. Hiç uygun veri yoksa 0. adımda/1. veri kontrolünde durulur. Eğitim birkaç dakika veya daha uzun sürebilir.

In [ ]:
if vad_split is None:
    print("VAD eğitimi atlandı: bu görev için uygun veri kümesi yok.")
else:
    vad_report = train_task(vad_split, TARGETS['vad'], '/kaggle/working/artifacts/vad', 'VAD', source_path=vad_path)
    print(json.dumps(vad_report['test_metrics'], ensure_ascii=False, indent=2))

In [ ]:
if big5_split is None:
    print("Big Five eğitimi atlandı: bu görev için uygun veri kümesi yok.")
else:
    big5_report = train_task(big5_split, TARGETS['big_five'], '/kaggle/working/artifacts/big-five', 'Big Five', source_path=big5_path)
    print(json.dumps(big5_report['test_metrics'], ensure_ascii=False, indent=2))

## Duygu polaritesi eğitimi

Bu zorunlu pilot, sabit sürümlü Türkçe veri kümesinde BERTurk'u ince ayarlar. Veri katılımcı kimliği içermediği için değerlendirme örnek düzeyindedir.

In [ ]:
import re, unicodedata, urllib.request
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding, Trainer, TrainingArguments

DATA_REVISION = 'ef87c7a2a7199a8b721c99425933316d35f9af2f'
DATASET = 'winvoker/turkish-sentiment-analysis-dataset'
BASE_URL = f'https://huggingface.co/datasets/{DATASET}/resolve/{DATA_REVISION}'
WORK = Path('/kaggle/working/digital_twin_sentiment')
WORK.mkdir(parents=True, exist_ok=True)
TRAIN_PATH, TEST_PATH = WORK / 'train.csv', WORK / 'test.csv'

for name, path in [('train.csv', TRAIN_PATH), ('test.csv', TEST_PATH)]:
    if not path.exists():
        try:
            urllib.request.urlretrieve(f'{BASE_URL}/{name}?download=true', path)
        except Exception as exc:
            raise RuntimeError(f'{name} indirilemedi. Kaggle Settings > Internet ayarını açıp Run All ile tekrar dene.') from exc

def canonical_label(value):
    text = unicodedata.normalize('NFKD', str(value)).casefold()
    text = ''.join(c for c in text if not unicodedata.combining(c)).replace('ı', 'i').strip()
    return {
        'positive': 'positive', 'pozitif': 'positive', 'olumlu': 'positive',
        'negative': 'negative', 'negatif': 'negative', 'olumsuz': 'negative',
        'neutral': 'neutral', 'notr': 'neutral', 'neutral sentiment': 'neutral'
    }.get(text)

def clean_frame(frame, name):
    if not {'text', 'label'}.issubset(frame.columns):
        raise ValueError(f'{name}: text ve label kolonları gerekli; bulunan={list(frame.columns)}')
    result = frame[['text', 'label']].copy()
    result['text'] = result['text'].fillna('').astype(str).str.replace(r'[\x00-\x1F\x7F]', ' ', regex=True).str.replace(r'\s+', ' ', regex=True).str.strip()
    result['label'] = result['label'].map(canonical_label)
    result = result[(result['text'] != '') & result['label'].notna()].copy()
    result['_key'] = result['text'].map(lambda x: unicodedata.normalize('NFKC', x).casefold())
    conflicts = result.groupby('_key')['label'].nunique()
    conflict_keys = set(conflicts[conflicts > 1].index)
    result = result[~result['_key'].isin(conflict_keys)].drop_duplicates('_key', keep='first').reset_index(drop=True)
    print(f'{name}: {len(result):,} temiz örnek; {len(conflict_keys):,} çelişkili metin çıkarıldı')
    return result

train_df = clean_frame(pd.read_csv(TRAIN_PATH), 'train')
test_df = clean_frame(pd.read_csv(TEST_PATH), 'test')
test_keys = set(test_df['_key'])
before = len(train_df)
train_df = train_df[~train_df['_key'].isin(test_keys)].reset_index(drop=True)
removed_overlap = before - len(train_df)
if train_df['label'].nunique() != 3 or test_df['label'].nunique() != 3:
    raise ValueError('Train ve test bölümlerinde üç sınıfın tamamı gerekli.')
if min(len(train_df), len(test_df)) < 30:
    raise ValueError('Veri bölümü eğitim için çok küçük.')
print('Test ile metin çakışması çıkarılan eğitim satırı:', removed_overlap)

def stratified_limit(frame, limit, seed=42):
    if len(frame) <= limit:
        return frame.sample(frac=1, random_state=seed).reset_index(drop=True)
    classes = sorted(frame['label'].unique())
    per_class = max(1, limit // len(classes))
    parts = [
        frame[frame['label'] == label].sample(n=min(len(frame[frame['label'] == label]), per_class), random_state=seed + i)
        for i, label in enumerate(classes)
    ]
    return pd.concat(parts).sample(frac=1, random_state=seed).reset_index(drop=True)

train_pool, validation_pool = train_test_split(train_df, test_size=0.1, random_state=42, stratify=train_df['label'])
train_pilot = stratified_limit(train_pool, 15000, 42)
validation_pilot = stratified_limit(validation_pool, 3000, 43)
test_pilot = stratified_limit(test_df, 3000, 44)
print({'train': len(train_pilot), 'validation': len(validation_pilot), 'test': len(test_pilot)})

LABEL2ID = {'negative': 0, 'neutral': 1, 'positive': 2}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}
MODEL_NAME = 'dbmdz/bert-base-turkish-cased'
MAX_LENGTH = 160
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class SentimentDataset(Dataset):
    def __init__(self, frame):
        self.texts = frame['text'].astype(str).tolist()
        self.labels = [LABEL2ID[x] for x in frame['label'].tolist()]
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, index):
        item = tokenizer(self.texts[index], truncation=True, max_length=MAX_LENGTH)
        item['labels'] = self.labels[index]
        return item

train_ds, validation_ds, test_ds = SentimentDataset(train_pilot), SentimentDataset(validation_pilot), SentimentDataset(test_pilot)
def sentiment_metrics(prediction):
    logits = prediction.predictions[0] if isinstance(prediction.predictions, tuple) else prediction.predictions
    pred = np.argmax(logits, axis=-1)
    return {
        'accuracy': float(accuracy_score(prediction.label_ids, pred)),
        'f1_macro': float(f1_score(prediction.label_ids, pred, average='macro', zero_division=0)),
        'f1_weighted': float(f1_score(prediction.label_ids, pred, average='weighted', zero_division=0))
    }

output_dir = Path('/kaggle/working/artifacts/sentiment')
output_dir.mkdir(parents=True, exist_ok=True)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3, label2id=LABEL2ID, id2label=ID2LABEL)
training_args = TrainingArguments(
    output_dir=str(output_dir / 'checkpoints'),
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    greater_is_better=True,
    save_total_limit=1,
    logging_steps=100,
    report_to='none',
    seed=42,
    fp16=torch.cuda.is_available()
)
trainer = Trainer(
    model=model, args=training_args, train_dataset=train_ds, eval_dataset=validation_ds,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=sentiment_metrics, tokenizer=tokenizer
)
trainer.train()
predicted_output = trainer.predict(test_ds, metric_key_prefix='test')
logits = predicted_output.predictions[0] if isinstance(predicted_output.predictions, tuple) else predicted_output.predictions
predictions = np.argmax(logits, axis=-1)
truth = predicted_output.label_ids
majority_label = train_pilot['label'].value_counts().idxmax()
majority_predictions = np.full(len(truth), LABEL2ID[majority_label], dtype=int)
baseline = {
    'majority_class': majority_label,
    'accuracy': float(accuracy_score(truth, majority_predictions)),
    'f1_macro': float(f1_score(truth, majority_predictions, average='macro', zero_division=0))
}
test_metrics = {
    'accuracy': float(accuracy_score(truth, predictions)),
    'f1_macro': float(f1_score(truth, predictions, average='macro', zero_division=0)),
    'f1_weighted': float(f1_score(truth, predictions, average='weighted', zero_division=0)),
    'majority_class_baseline': baseline,
    'per_class': classification_report(truth, predictions, labels=[0, 1, 2], target_names=[ID2LABEL[i] for i in range(3)], output_dict=True, zero_division=0),
    'confusion_matrix_negative_neutral_positive': confusion_matrix(truth, predictions, labels=[0, 1, 2]).tolist()
}
trainer.save_model(str(output_dir / 'model'))
tokenizer.save_pretrained(str(output_dir / 'model'))
report = {
    'task': 'Turkish sentiment polarity: negative/neutral/positive',
    'model': MODEL_NAME,
    'dataset': DATASET,
    'dataset_revision': DATA_REVISION,
    'dataset_license_declared_by_hub': 'CC BY-SA 4.0',
    'dataset_reference': f'https://huggingface.co/datasets/{DATASET}',
    'seed': 42,
    'max_length': MAX_LENGTH,
    'epochs': 2,
    'batch_size': 8,
    'train_rows': len(train_pilot),
    'validation_rows': len(validation_pilot),
    'test_rows': len(test_pilot),
    'training_rows_removed_due_to_test_overlap': int(removed_overlap),
    'train_source_sha256': hashlib.sha256(TRAIN_PATH.read_bytes()).hexdigest(),
    'test_source_sha256': hashlib.sha256(TEST_PATH.read_bytes()).hexdigest(),
    'test_metrics': test_metrics,
    'scientific_limitations': [
        'Polarite sınıflandırması sürekli VAD değildir.',
        'Bu deney Big Five kişilik tahmini veya birey düzeyinde dijital ikiz doğrulaması değildir.',
        'Kaynak veri kümesinde katılımcı kimlikleri bulunmadığından test örnek düzeyindedir.',
        'Hub lisans beyanı, birleşik verideki alt kaynakların kullanım hakkını tek başına kanıtlamaz.'
    ]
}
metrics_path = output_dir / 'metrics.json'
metrics_path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps({'metrics_path': str(metrics_path), 'test_metrics': test_metrics}, ensure_ascii=False, indent=2))

In [ ]:
import shutil
artifact_root = Path('/kaggle/working/artifacts')
if not artifact_root.is_dir() or not list(artifact_root.rglob('metrics.json')):
    raise FileNotFoundError('Metrik dosyası yok; başarılı bir eğitim görevi tamamlanmadı.')
archive = shutil.make_archive('/kaggle/working/dijital-ikiz-model-artifacts', 'zip', str(artifact_root))
print('Model paketi:', archive)
for metrics_path in sorted(artifact_root.rglob('metrics.json')):
    print('\n', metrics_path.relative_to(artifact_root))
    print(metrics_path.read_text(encoding='utf-8'))

## Sonuçlar

Her görev yalnızca kendi gerçek test metriğiyle raporlanır. Polarite, VAD ve Big Five sonuçları birbirinin yerine kullanılamaz. Paket: `dijital-ikiz-model-artifacts.zip`.